# RL Constrained CQL Training

This notebook:
1. Loads offline RL tensors from `data/processed/rl_tensors_2022_2023_constrained.npz`
2. Loads CQL hyperparameters from `configs/model.yaml` and `configs/training.yaml`
3. Trains a dueling CQL using `src/rl/cql.py`
4. Saves the trained model to `models/`

## 1. Imports & Paths

In [1]:
import os
import sys
from pathlib import Path
import torch
import yaml

PROJECT_ROOT = Path(os.getcwd()).resolve().parent
sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\Matth\OneDrive\Desktop\CS3346\MLB-Bullpen-Strategy


In [2]:
from src.rl.cql import (
    load_cql_training_config,
    train_cql,
    BullpenOfflineDataset,
    RLDatasetConfig,
)
from src.ope.offline_eval_cql import (
    OfflineEvalConfig,
    load_model_and_dataset,
    evaluate_td_error_full_mse,
    direct_policy_value_estimate,
    compute_policy_behavior_stats,
    compute_q_distributions,
    summarize_policy_behavior_stats,
    summarize_q_distributions,
)

## 2. Configurations

In [3]:
DATA_DIR = PROJECT_ROOT / "data"
PROC_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

YEAR_TAG = "2022_2023"
RL_TENSORS_PATH = PROC_DIR / f"rl_tensors_{YEAR_TAG}_constrained.npz"
MODEL_CFG_PATH = CONFIG_DIR / "model.yaml"
TRAIN_CFG_PATH = CONFIG_DIR / "training.yaml"
MODEL_OUT_PATH = MODELS_DIR / f"constrained_cql_model_{YEAR_TAG}.pt"

print("RL tensors:", RL_TENSORS_PATH)
print("Model config:", MODEL_CFG_PATH)
print("Training config:", TRAIN_CFG_PATH)
print("Model output:", MODEL_OUT_PATH)

RL tensors: C:\Users\Matth\OneDrive\Desktop\CS3346\MLB-Bullpen-Strategy\data\processed\rl_tensors_2022_2023_constrained.npz
Model config: C:\Users\Matth\OneDrive\Desktop\CS3346\MLB-Bullpen-Strategy\configs\model.yaml
Training config: C:\Users\Matth\OneDrive\Desktop\CS3346\MLB-Bullpen-Strategy\configs\training.yaml
Model output: C:\Users\Matth\OneDrive\Desktop\CS3346\MLB-Bullpen-Strategy\models\constrained_cql_model_2022_2023.pt


## 3. Load Dataset & Build Model

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

train_cfg = load_cql_training_config(
    model_config_path=MODEL_CFG_PATH,
    data_path=RL_TENSORS_PATH,
    device=device,
)

train_cfg

ds = BullpenOfflineDataset(
    RLDatasetConfig(
        data_path=train_cfg.data_path,
        device=train_cfg.device,
    )
)

print("Dataset size:", len(ds))
print("State dim:", ds.state_dim)
print("Num actions:", ds.num_actions)
print("H (next hitters window):", ds.H)
print("R (max relievers per team):", ds.R)

Using device: cpu
Dataset size: 22249
State dim: 208
Num actions: 11
H (next hitters window): 5
R (max relievers per team): 10


## 4. Create Dueling CQL Model + Trainer
This calls train_dqn(train_cfg), which:

* loads BullpenOfflineDataset from train_cfg.data_path
* splits into train/val by train_cfg.val_fraction
* trains a dueling CQL with a target network
* logs TD-error periodically using evaluate_td_error in cql.py

In [5]:
constrained_cql_model = train_cql(train_cfg)

[CQL] step=0 loss=4111.87939
      val_td_error=3632.13143
      (new best val TD: 3632.13143)
[CQL] step=1000 loss=52.98056
      val_td_error=23.78140
      (new best val TD: 23.78140)
[CQL] step=2000 loss=20.21912
      val_td_error=10.90344
      (new best val TD: 10.90344)
[CQL] step=3000 loss=13.21926
      val_td_error=8.44740
      (new best val TD: 8.44740)
[CQL] step=4000 loss=9.63261
      val_td_error=6.88912
      (new best val TD: 6.88912)
[CQL] step=5000 loss=7.38393
      val_td_error=5.08653
      (new best val TD: 5.08653)
[CQL] step=6000 loss=8.79424
      val_td_error=4.41152
      (new best val TD: 4.41152)
[CQL] step=7000 loss=4.67509
      val_td_error=3.53304
      (new best val TD: 3.53304)
[CQL] step=8000 loss=6.11576
      val_td_error=2.99743
      (new best val TD: 2.99743)
[CQL] step=9000 loss=3.23616
      val_td_error=2.47699
      (new best val TD: 2.47699)
[CQL] step=10000 loss=2.48341
      val_td_error=2.08329
      (new best val TD: 2.08329)
[CQL] s

## 5. Save trained model weights

In [6]:
torch.save(constrained_cql_model.state_dict(), MODEL_OUT_PATH)
MODEL_OUT_PATH

WindowsPath('C:/Users/Matth/OneDrive/Desktop/CS3346/MLB-Bullpen-Strategy/models/constrained_cql_model_2022_2023.pt')

## Offline Policy Evaluation (OPE)
Now we use src/ope/offline_eval.py to:

* load the saved model and dataset
* compute:
    * Mean Squared TD Error (MSTE)
    * Direct Q-based value of the greedy policy
    * Action agreement with the logged policy

In [7]:
ope_cfg = OfflineEvalConfig(
    model_config_path=MODEL_CFG_PATH,
    model_path=MODEL_OUT_PATH,
    tensors_path=RL_TENSORS_PATH,
    device=device,
    batch_size=2048,
    gamma=train_cfg.gamma,
)

eval_model, eval_ds, eval_loader = load_model_and_dataset(ope_cfg)

print("Eval dataset size:", len(eval_ds))
print("State dim:", eval_ds.state_dim)
print("Num actions:", eval_ds.num_actions)

Eval dataset size: 22249
State dim: 208
Num actions: 11


## 6. Mean Squared TD Error (MSTE)
This is the mean squared Bellman residual over the full dataset. It reuses evaluate_td_error from cql.py under the hood, passing model as both the online and target networks.

In [8]:
mste = evaluate_td_error_full_mse(
    model=eval_model,
    loader=eval_loader,
    gamma=ope_cfg.gamma,
    device=ope_cfg.device,
)

print(f"Mean Squared TD Error (MSTE): {mste:.6f}")

Mean Squared TD Error (MSTE): 0.576098


## 7. Direct Q-based value estimate (FQE-style Direct Method)
For each state s: - compute Q(s, a) for all actions - mask unavailable actions - take greedy action a* = argmax_a Q(s, a) - define V_hat(s) = Q(s, a*)

Then average V_hat(s) across the dataset as an estimate of V(pi_greedy).

In [9]:
dm_value = direct_policy_value_estimate(
    model=eval_model,
    loader=eval_loader,
    device=ope_cfg.device,
)

print(f"Direct Q-based value estimate (V(pi_greedy)): {dm_value:.6f}")

Direct Q-based value estimate (V(pi_greedy)): 5.999926


## 8. Policy Behavior Stats and Q distributions
How often does the greedy CQL action (respecting availability mask) match the logged (historical) action from the dataset?

In [10]:
# New distributional metrics
policy_stats = compute_policy_behavior_stats(eval_model, eval_loader, device=ope_cfg.device)

q_stats = compute_q_distributions(eval_model, eval_loader, device=ope_cfg.device)

## 9. Summary

In [11]:
print("========= FINAL CQL EVALUATION RESULTS =========")
print(f"TD Error (MSTE):              {mste:.6f}")
print(f"Direct Q-based V(pi_greedy):  {dm_value:.6f}")
summarize_policy_behavior_stats(policy_stats)
summarize_q_distributions(q_stats)

========= FINAL CQL EVALUATION RESULTS =========
TD Error (MSTE):              0.576098
Direct Q-based V(pi_greedy):  5.999926
=== Policy vs Behavior Stats ===
Num samples:   22249
Num actions:   11

Behavior pull rate: 20.23%
Policy pull rate:   90.79%
Action agreement:   9.01%

Behavior action counts (per action index):
[17749   643   687   596   579   410   434   369   308   270   204]
Policy action counts (per action index):
[2050 4547  172 1009  292 3152  927 1863 3909 2442 1886]
Valid action counts (per action index):
[22249 21922 21209 21561 21081 21469 20782 21433 20896 20575 20860]
=== Q Distribution Stats ===
q_all_valid: n=234037, mean=5.794, std=3.031, min=-0.543, max=33.145
q_stay: n=22249, mean=5.745, std=2.924, min=-0.543, max=28.859
q_best_pull: n=22249, mean=5.996, std=3.126, min=0.880, max=33.145
q_stay_minus_best_pull: n=22249, mean=-0.252, std=0.256, min=-4.286, max=0.272


In [12]:
import numpy as np
from pathlib import Path

npz = np.load(Path("../data/processed/rl_tensors_2022_2023.npz"))

for key in ["reward_folded"]:
    x = npz[key]
    print(key, "shape:", x.shape)
    print(
        key,
        "mean:", float(x.mean()),
        "std:", float(x.std()),
        "min:", float(x.min()),
        "max:", float(x.max()),
    )

reward_folded shape: (407660,)
reward_folded mean: -0.010023724287748337 std: 0.6859701871871948 min: -7.711379528045654 max: 1.1493159532546997
